# BraTS 2024 — Exploratory Data Analysis

Notebook sections:

1. Colab setup
2. Imports, paths and configuration
3. Dataset overview (GLI / MEN)
4. Single-patient exploration (GLI)
5. Brain z-boundaries and slice extraction
6. GLI segmentation analysis — 100-patient random sample
7. MEN single-patient exploration

## 1. Colab setup

In [ ]:
# Clone repo (if needed), switch to working branch, install package
!test -d /content/BraTS_thesis || git clone https://github.com/AdaBro24/BraTS_thesis.git /content/BraTS_thesis
!git -C /content/BraTS_thesis checkout extra_channel_norm
!git -C /content/BraTS_thesis pull origin extra_channel_norm
!pip install -q -e /content/BraTS_thesis

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Imports, paths and configuration

In [ ]:
import sys
import json
import random
from collections import Counter
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from tqdm.auto import tqdm

sys.path.insert(0, "/content/BraTS_thesis/src")

In [ ]:
ZIP_DIR = Path("/content/drive/MyDrive/engineering_thesis/brats")

DATA_ROOT = ZIP_DIR / "unpacked"
GLI_ROOT = DATA_ROOT / "BraTS-GLI" / "training_data1_v2"
MEN_ROOT = DATA_ROOT / "BraTS-MEN" / "BraTS-MEN-RT-Train-v2"

Z_BOUNDARIES_PATH = DATA_ROOT / "patients_z_boundaries.json"
REPO_BOUNDARIES_PATH = Path("/content/BraTS_thesis/data/patients_z_boundaries.json")

In [ ]:
MODALITY_SUFFIXES = {
    "T1": "t1n",
    "T1CE": "t1c",
    "T2": "t2w",
    "FLAIR": "t2f",
    "SEG": "seg",
}

SEG_LABELS = {
    1: "NETC",
    2: "SNFH",
    3: "ET",
    4: "RC",
}

SEG_COLORS = {
    1: "#7B2CBF",
    2: "#1F9E89",
    3: "#57C95A",
    4: "#FDE725",
}

SEG_CMAP = ListedColormap([SEG_COLORS[label] for label in SEG_LABELS])
SEG_NORM = BoundaryNorm(boundaries=[0.5, 1.5, 2.5, 3.5, 4.5], ncolors=SEG_CMAP.N)
SEG_LEGEND = [
    Patch(facecolor=SEG_COLORS[label], label=f"{label}: {name}")
    for label, name in SEG_LABELS.items()
]

In [ ]:
def find_patient_files(patient_dir: Path, modality_suffixes=None) -> dict:
    if modality_suffixes is None:
        modality_suffixes = MODALITY_SUFFIXES

    patient_files = {}
    for file_path in patient_dir.iterdir():
        if not file_path.is_file():
            continue

        name = file_path.name.replace(".nii.gz", "").replace(".nii", "").lower()
        for modality, suffix in modality_suffixes.items():
            if name.endswith(suffix):
                patient_files[modality] = file_path
                break

    return patient_files


def list_patient_dirs(data_dir: Path) -> list[Path]:
    return [entry for entry in data_dir.iterdir() if entry.is_dir()]

## 3. Dataset overview (GLI / MEN)

In [ ]:
gli_patients = list_patient_dirs(GLI_ROOT)
men_patients = list_patient_dirs(MEN_ROOT)

print(f"Total GLI patients: {len(gli_patients)}")
print(f"Total MEN patients: {len(men_patients)}")

In [ ]:
required_modalities = set(MODALITY_SUFFIXES)

for dataset_name, data_root in [("GLI", GLI_ROOT), ("MEN", MEN_ROOT)]:
    patient_dirs = list_patient_dirs(data_root)
    incomplete = {
        p.name: required_modalities - set(find_patient_files(p))
        for p in patient_dirs
        if required_modalities - set(find_patient_files(p))
    }

    print(f"\n{dataset_name}")
    print(f"Number of patients: {len(patient_dirs)}")
    print(f"Number of incomplete patients: {len(incomplete)}")

    for patient_id, missing in list(incomplete.items())[:5]:
        print(f"  {patient_id}: missing {sorted(missing)}")
    if len(incomplete) > 5:
        print(f"  ... and {len(incomplete) - 5} more")

In [ ]:
random.seed(42)
sampled_gli_patients = random.sample(list_patient_dirs(GLI_ROOT), k=100)

shape_counts = Counter(
    nib.load(file_path).shape
    for patient_dir in sampled_gli_patients
    for file_path in patient_dir.glob("*.nii.gz")
)

print("Unique shapes in 100-patient sample:", dict(shape_counts))

In [ ]:
from brain_mri.data.nifti_loader import load_patient

sample_dir = list_patient_dirs(GLI_ROOT)[0]
sample_files = find_patient_files(sample_dir)

print("Patient:", sample_dir.name)

images, segmentation, brain = load_patient(sample_dir)
print("Images shape:", images.shape, "| dtype:", images.dtype)
print("SEG shape:", segmentation.shape, "| dtype:", segmentation.dtype)
print("SEG labels:", np.unique(segmentation))

In [ ]:
for modality, file_path in sample_files.items():
    print(f"{modality}: shape={nib.load(file_path).shape}")

## 4. Single-patient exploration (GLI)

In [ ]:
modalities = ["T1", "T1CE", "T2", "FLAIR", "SEG"]
seg_volume = nib.load(sample_files["SEG"]).get_fdata(dtype=np.float32)
slice_idx = np.argmax(np.sum(seg_volume, axis=(0, 1)))

fig, axes = plt.subplots(1, len(modalities), figsize=(20, 4))
for ax, modality in zip(axes, modalities):
    volume = nib.load(sample_files[modality]).get_fdata(dtype=np.float32)

    if modality == "SEG":
        masked_seg = np.ma.masked_where(volume[:, :, slice_idx] == 0, volume[:, :, slice_idx])
        ax.imshow(masked_seg, cmap=SEG_CMAP, norm=SEG_NORM)
    else:
        ax.imshow(volume[:, :, slice_idx], cmap="gray")

    ax.set_title(f"{modality}, z={slice_idx}")
    ax.axis("off")

fig.suptitle(f"Patient: {sample_dir.name}")
plt.tight_layout()
plt.show()

In [ ]:
base_modality = "FLAIR"

base_volume = nib.load(sample_files[base_modality]).get_fdata(dtype=np.float32)
slice_idx = base_volume.shape[2] // 2

base_slice = base_volume[:, :, slice_idx]
seg_slice = seg_volume[:, :, slice_idx]
masked_seg = np.ma.masked_where(seg_slice == 0, seg_slice)

plt.figure(figsize=(7, 7))
plt.imshow(base_slice, cmap="gray")
plt.imshow(masked_seg, cmap=SEG_CMAP, norm=SEG_NORM, alpha=1.0)
plt.title(f"{base_modality} + SEG, z={slice_idx}")
plt.axis("off")
plt.legend(handles=SEG_LEGEND, loc="upper right", title="SEG labels")
plt.show()

In [ ]:
for modality in ["FLAIR", "T1", "T1CE", "T2"]:
    volume = nib.load(sample_files[modality]).get_fdata(dtype=np.float32)
    non_zero_voxels = volume[volume != 0]

    print(f"\n{modality}")
    print(f"shape: {volume.shape}")
    print(f"min: {volume.min()}")
    print(f"max: {volume.max()}")
    print(f"mean: {volume.mean()}")
    print(f"median: {np.median(volume)}")
    print(f"std: {volume.std()}")
    print(f"quantiles [0.01, 0.25, 0.5, 0.75, 0.99]: {np.quantile(non_zero_voxels, [0.01, 0.25, 0.5, 0.75, 0.99])}")
    print(f"zero voxels: {(volume == 0).sum()}")
    print(f"nonzero voxels: {len(non_zero_voxels)}")
    print(f"NaN: {np.isnan(volume).sum()}")
    print(f"inf: {np.isinf(volume).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, modality in zip(axes, ["FLAIR", "T1", "T1CE", "T2"]):
    volume = nib.load(sample_files[modality]).get_fdata(dtype=np.float32)
    non_zero_voxels = volume[volume != 0]

    ax.hist(non_zero_voxels, bins=100)
    ax.set_title(modality)
    ax.set_xlabel("Intensity")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
labels, counts = np.unique(seg_volume, return_counts=True)
for label, count in zip(labels, counts):
    print(f"Label: {label}, Count: {count}")

tumour_voxels_per_slice = np.sum(seg_volume != 0, axis=(0, 1))
best_seg_slice = np.argmax(tumour_voxels_per_slice)
print(f"Slice with the most tumour voxels: {best_seg_slice}")

In [ ]:
base_modality = "FLAIR"
slice_idx = 107

base_slice = base_volume[:, :, slice_idx]
seg_slice = seg_volume[:, :, slice_idx]
masked_seg = np.ma.masked_where(seg_slice == 0, seg_slice)

plt.figure(figsize=(7, 7))
plt.imshow(base_slice, cmap="gray")
plt.imshow(masked_seg, cmap=SEG_CMAP, norm=SEG_NORM, alpha=1.0)
plt.title(f"{base_modality} + SEG, z={slice_idx}")
plt.axis("off")
plt.legend(handles=SEG_LEGEND, loc="upper right", title="SEG labels")
plt.show()

In [ ]:
sample2_dir = list_patient_dirs(GLI_ROOT)[1]
sample2_files = find_patient_files(sample2_dir)

flair2 = nib.load(sample2_files["FLAIR"]).get_fdata(dtype=np.float32)
seg2 = nib.load(sample2_files["SEG"]).get_fdata(dtype=np.float32)

et_per_slice = np.sum(seg2 == 3, axis=(0, 1))
slice_idx = np.argmax(et_per_slice)
masked_seg = np.ma.masked_where(seg2[:, :, slice_idx] == 0, seg2[:, :, slice_idx])

plt.figure(figsize=(7, 7))
plt.imshow(flair2[:, :, slice_idx], cmap="gray")
plt.imshow(masked_seg, cmap=SEG_CMAP, norm=SEG_NORM, alpha=1.0)
plt.title(f"FLAIR + SEG (ET-dense), z={slice_idx}, patient: {sample2_dir.name}")
plt.axis("off")
plt.legend(handles=SEG_LEGEND, loc="upper right", title="SEG labels")
plt.show()

print("ET voxels:", (seg2 == 3).sum())

In [ ]:
slice_indices = np.linspace(0, flair2.shape[2] - 1, 18, dtype=int)
fig, axes = plt.subplots(3, 6, figsize=(20, 12))
axes = axes.flatten()

for ax, z_idx in zip(axes, slice_indices):
    ax.imshow(flair2[:, :, z_idx].T, cmap="gray", origin="lower")
    ax.set_title(f"Slice {z_idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Brain z-boundaries and slice extraction

Scan FLAIR volumes to record the first/last axial slice containing brain tissue.
Results are saved to `patients_z_boundaries.json` and consumed by the slice
extraction pipeline. Takes ~4 min for 1350 GLI patients.

In [ ]:
patient_dirs = list_patient_dirs(GLI_ROOT)
patients_z_boundaries = {}

for p_dir in tqdm(patient_dirs, desc="Scanning Z axis"):
    patient_files = find_patient_files(p_dir)

    if "FLAIR" not in patient_files:
        continue

    volume = nib.load(patient_files["FLAIR"]).get_fdata(dtype=np.float32)

    z_has_brain = np.any(volume > 0, axis=(0, 1))
    brain_indices = np.where(z_has_brain)[0]

    patients_z_boundaries[p_dir.name] = [
        int(brain_indices[0]),
        int(brain_indices[-1]),
    ]

with open(Z_BOUNDARIES_PATH, "w") as f:
    json.dump(patients_z_boundaries, f)

print(f"Saved {len(patients_z_boundaries)} patients to {Z_BOUNDARIES_PATH}")

In [ ]:
from brain_mri.slice_extraction import extract_patient_slices

patient_dir = GLI_ROOT / "BraTS-GLI-00005-100"
slices = extract_patient_slices(patient_dir, REPO_BOUNDARIES_PATH)

print(slices[0]["image"].shape)
print(slices[0]["mask"].shape)

In [ ]:
best_slice = max(slices, key=lambda s: (s["mask"] != 0).sum())

fig, axes = plt.subplots(1, 6, figsize=(18, 4))
channel_names = ["T1", "T1CE", "T2", "FLAIR", "brain"]

for i, name in enumerate(channel_names):
    axes[i].imshow(best_slice["image"][i], cmap="gray")
    axes[i].set_title(name)
    axes[i].axis("off")

axes[5].imshow(best_slice["mask"], cmap="viridis")
axes[5].set_title("mask")
axes[5].axis("off")
plt.show()

In [ ]:
arr = best_slice["image"][0]
print("shape:", arr.shape)
print("min:", arr.min(), "max:", arr.max())
print("mean:", arr.mean(), "std:", arr.std())
print("zeros:", (arr == 0).sum(), "/", arr.size)
print("first unique values:", np.unique(arr)[:10])

In [ ]:
print("min pixels count:", (arr == arr.min()).sum())

ys, xs = np.where(arr == arr.min())
print("first xy:", list(zip(ys[:10], xs[:10])))

plt.imshow(arr == arr.min(), cmap="gray")
plt.title("Min value locations")
plt.show()

In [ ]:
normalised_flair = images[3]
brain_voxels = normalised_flair[normalised_flair != 0]

print("Mean:", brain_voxels.mean())
print("Std:", brain_voxels.std())
print("Background values:", np.unique(normalised_flair[normalised_flair == 0]))

## 6. GLI segmentation analysis — 100-patient random sample

Uses `sampled_gli_patients` from section 3 (seed 42).

In [ ]:
tumour_records = []

for patient_dir in sampled_gli_patients:
    patient_files = find_patient_files(patient_dir)
    seg = nib.load(patient_files["SEG"]).get_fdata(dtype=np.float32)

    labels, counts = np.unique(seg, return_counts=True)
    label_counts = dict(zip(labels.astype(int), counts))

    total_voxels = seg.size
    whole_tumour_voxels = int(np.isin(seg, [1, 2, 3]).sum())

    tumour_records.append({
        "patient_dir": patient_dir,
        "total_voxels": total_voxels,
        "segmented_voxels": int((seg != 0).sum()),
        "whole_tumour_voxels": whole_tumour_voxels,
        "tumour_core_voxels": int(np.isin(seg, [1, 3]).sum()),
        "resection_cavity_voxels": int((seg == 4).sum()),
        "whole_tumour_percent": 100 * whole_tumour_voxels / total_voxels,
        "label_0": label_counts.get(0, 0),
        "label_1": label_counts.get(1, 0),
        "label_2": label_counts.get(2, 0),
        "label_3": label_counts.get(3, 0),
        "label_4": label_counts.get(4, 0),
    })

tumour_df = pd.DataFrame(tumour_records)

In [ ]:
tumour_df[[
    "segmented_voxels",
    "whole_tumour_voxels",
    "tumour_core_voxels",
    "resection_cavity_voxels",
    "whole_tumour_percent",
    "label_1",
    "label_2",
    "label_3",
    "label_4",
]].describe()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(tumour_df["whole_tumour_voxels"], bins=20, edgecolor="black")
plt.xlabel("Whole tumour voxels")
plt.ylabel("Frequency")
plt.title("Distribution of whole tumour volume")
plt.show()

In [ ]:
class_columns = ["label_1", "label_2", "label_3", "label_4"]

class_totals = tumour_df[class_columns].sum()
class_presence = (tumour_df[class_columns] > 0).sum(axis=0)

print("Total voxels per class:")
print(class_totals)
print("Patients containing each class:")
print(class_presence)

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(class_totals.index, class_totals.values, edgecolor="black")
plt.xlabel("Segmentation class")
plt.ylabel("Total voxels across 100 patients")
plt.title("GLI segmentation class distribution")
plt.show()

In [ ]:
for label in [1, 2, 3, 4]:
    tumour_df[f"label_{label}_within_segmentation_percent"] = (
        100 * tumour_df[f"label_{label}"] / tumour_df["segmented_voxels"]
    )

within_segmentation_columns = [
    f"label_{label}_within_segmentation_percent"
    for label in [1, 2, 3, 4]
]

tumour_df[within_segmentation_columns].describe()

In [ ]:
tumour_df["segmentation_percentage_sum"] = tumour_df[within_segmentation_columns].sum(axis=1)
tumour_df["segmentation_percentage_sum"].describe()

In [ ]:
plt.figure(figsize=(10, 6))
plt.boxplot(tumour_df[within_segmentation_columns])
plt.xticks(ticks=range(1, 5), labels=["NETC", "SNFH", "ET", "RC"])
plt.ylabel("Percentage within segmented region")
plt.title("Distribution of segmentation labels")
plt.show()

In [ ]:
cases = {
    "smallest": tumour_df.loc[tumour_df["whole_tumour_voxels"].idxmin()],
    "largest": tumour_df.loc[tumour_df["whole_tumour_voxels"].idxmax()],
}

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (case_name, case_data) in zip(axes, cases.items()):
    patient_files = find_patient_files(case_data["patient_dir"])

    flair_volume = nib.load(patient_files["FLAIR"]).get_fdata(dtype=np.float32)
    seg = nib.load(patient_files["SEG"]).get_fdata(dtype=np.float32)

    best_z = np.argmax(np.sum(seg, axis=(0, 1)))
    masked_seg = np.ma.masked_where(seg[:, :, best_z] == 0, seg[:, :, best_z])

    ax.imshow(flair_volume[:, :, best_z], cmap="gray")
    ax.imshow(masked_seg, cmap=SEG_CMAP, norm=SEG_NORM, alpha=1.0)
    ax.set_title(
        f"{case_name}\n"
        f"{case_data['patient_dir'].name}\n"
        f"Whole tumour: {case_data['whole_tumour_voxels']:,} voxels"
    )
    ax.axis("off")

fig.legend(handles=SEG_LEGEND, loc="upper center", ncol=4, title="SEG labels")
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

In [ ]:
intensity_records = []

for patient_dir in sampled_gli_patients:
    patient_files = find_patient_files(patient_dir)

    for modality in ["FLAIR", "T1", "T1CE", "T2"]:
        volume = nib.load(patient_files[modality]).get_fdata(dtype=np.float32)
        non_zero_voxels = volume[volume > 0]

        intensity_records.append({
            "patient_dir": patient_dir.name,
            "modality": modality,
            "mean_intensity": non_zero_voxels.mean(),
            "median_intensity": np.median(non_zero_voxels),
            "std_intensity": non_zero_voxels.std(),
            "min_intensity": non_zero_voxels.min(),
            "max_intensity": non_zero_voxels.max(),
            "q01_intensity": np.quantile(non_zero_voxels, 0.01),
            "q99_intensity": np.quantile(non_zero_voxels, 0.99),
        })

intensity_df = pd.DataFrame(intensity_records)
intensity_df.head()

In [ ]:
intensity_summary = (
    intensity_df
    .groupby("modality")
    .agg(
        patients=("patient_dir", "count"),
        mean_intensity_mean=("mean_intensity", "mean"),
        mean_intensity_std=("mean_intensity", "std"),
        q01_mean=("q01_intensity", "mean"),
        q99_mean=("q99_intensity", "mean"),
    )
    .round(2)
)

intensity_summary

## 7. MEN single-patient exploration

MEN-RT patients contain only `T1CE` and a `GTV` mask (no full 4-modality set).

In [ ]:
MEN_SUFFIXES = {**MODALITY_SUFFIXES, "GTV": "gtv"}

men_sample_dir = list_patient_dirs(MEN_ROOT)[0]
men_sample_files = find_patient_files(men_sample_dir, MEN_SUFFIXES)

print("Sample directory:", men_sample_dir)
print("Sample files:", men_sample_files)

In [ ]:
men_volume = nib.load(men_sample_files["T1CE"]).get_fdata(dtype=np.float32)
slice_idx = np.argmax(np.sum(men_volume, axis=(0, 1)))

plt.imshow(men_volume[:, :, slice_idx], cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
MEN_MODALITIES = ["T1CE", "GTV"]
gtv_volume = nib.load(men_sample_files["GTV"]).get_fdata(dtype=np.float32)
slice_idx = np.argmax(np.sum(gtv_volume != 0, axis=(0, 1)))

fig, axes = plt.subplots(1, len(MEN_MODALITIES), figsize=(10, 5))
for ax, modality in zip(axes, MEN_MODALITIES):
    volume = nib.load(men_sample_files[modality]).get_fdata(dtype=np.float32)
    cmap = "viridis" if modality == "GTV" else "gray"

    ax.imshow(volume[:, :, slice_idx], cmap=cmap)
    ax.set_title(f"{modality}, z={slice_idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
base_modality = "T1CE"
base_volume = nib.load(men_sample_files[base_modality]).get_fdata(dtype=np.float32)

slice_idx = np.argmax(np.sum(gtv_volume != 0, axis=(0, 1)))
masked_gtv = np.ma.masked_where(gtv_volume[:, :, slice_idx] == 0, gtv_volume[:, :, slice_idx])

plt.figure(figsize=(7, 7))
plt.imshow(base_volume[:, :, slice_idx], cmap="gray")
plt.imshow(masked_gtv, cmap="viridis", alpha=0.5)
plt.title(f"{base_modality} + GTV, z={slice_idx}")
plt.axis("off")
plt.show()